<a href="https://colab.research.google.com/github/anujjakhotiya/AI-DL_2026/blob/main/Lab_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

from google.colab import files
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ============================================================
# 2. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


# ============================================================
# 3. UPLOAD DATASET ZIP
# ============================================================

uploaded = files.upload()

zip_file = list(uploaded.keys())[0]

print("Uploaded:", zip_file)


# ============================================================
# 4. EXTRACT DATASET
# ============================================================

extract_path = "/content/beans_dataset"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

os.makedirs(extract_path)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")


# ============================================================
# 5. FIND TRAIN / VALIDATION / TEST FOLDERS
# ============================================================

train_dir = None
val_dir = None
test_dir = None

class_names = {
    "angular_leaf_spot",
    "bean_rust",
    "healthy"
}

# Iterate through all subdirectories in the extracted path
for root, dirs, _ in os.walk(extract_path):
    # Check if 'train' is one of the directories at the current 'root' level
    if "train" in dirs and train_dir is None:
        # Construct the full path to the potential 'train' directory
        candidate_train_dir = os.path.join(root, "train")
        # Check if this 'train' directory contains at least one of the expected class names
        # This helps confirm it's the correct 'train' folder for the dataset.
        if os.path.isdir(candidate_train_dir) and any(cls_name in os.listdir(candidate_train_dir) for cls_name in class_names):
            train_dir = candidate_train_dir

    # Check for 'validation' or 'val'
    if ("validation" in dirs or "val" in dirs) and val_dir is None:
        candidate_val_dir = None
        if "validation" in dirs:
            candidate_val_dir = os.path.join(root, "validation")
        elif "val" in dirs:
            candidate_val_dir = os.path.join(root, "val")

        if candidate_val_dir and os.path.isdir(candidate_val_dir) and any(cls_name in os.listdir(candidate_val_dir) for cls_name in class_names):
            val_dir = candidate_val_dir

    # Check for 'test'
    if "test" in dirs and test_dir is None:
        candidate_test_dir = os.path.join(root, "test")
        if os.path.isdir(candidate_test_dir) and any(cls_name in os.listdir(candidate_test_dir) for cls_name in class_names):
            test_dir = candidate_test_dir

    # Optimization: if all relevant directories are found, we can stop walking
    if train_dir and (val_dir or test_dir or (not "validation" in dirs and not "val" in dirs and not "test" in dirs)):
        break


print("Train folder:", train_dir)
print("Validation folder:", val_dir)
print("Test folder:", test_dir)


# ============================================================
# 6. DATA AUGMENTATION
# ============================================================

train_transform = transforms.Compose([

    transforms.Resize((128, 128)),

    # Data augmentation
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Validation/Test should NOT use random augmentation

test_transform = transforms.Compose([

    transforms.Resize((128, 128)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# 7. LOAD DATASETS
# ============================================================

if train_dir is None:
    raise FileNotFoundError(
        "Training folder not found. Check the ZIP file structure."
    )

train_dataset = datasets.ImageFolder(
    train_dir,
    transform=train_transform
)

if val_dir is not None:

    val_dataset = datasets.ImageFolder(
        val_dir,
        transform=test_transform
    )

else:

    # If validation folder is unavailable,
    # create validation split from training data

    full_dataset = datasets.ImageFolder(
        train_dir,
        transform=train_transform
    )

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )


# Test dataset

if test_dir is not None:

    test_dataset = datasets.ImageFolder(
        test_dir,
        transform=test_transform
    )

else:

    test_dataset = val_dataset


print("\nClasses:", train_dataset.dataset.classes
      if isinstance(train_dataset, torch.utils.data.Subset)
      else train_dataset.classes)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Testing images:", len(test_dataset))


# ============================================================
# 8. DATA LOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)


# ============================================================
# 9. CNN MODEL
# ============================================================

class CNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(256 * 8 * 8, 256),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256, 3)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


model = CNN().to(device)

print("\nModel:")
print(model)


# ============================================================
# 10. LOSS FUNCTION AND OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


# ============================================================
# 11. TRAINING
# ============================================================

epochs = 10

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []


for epoch in range(epochs):

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()


    train_loss = running_loss / len(train_loader)

    train_accuracy = 100 * correct / total


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()


    val_loss = running_loss / len(val_loader)

    val_accuracy = 100 * correct / total


    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)


    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_accuracy:.2f}%"
    )


# ============================================================
# 12. FINAL TEST EVALUATION
# ============================================================

model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, predictions = torch.max(outputs, 1)

        all_labels.extend(labels.numpy())

        all_predictions.extend(
            predictions.cpu().numpy()
        )


accuracy = accuracy_score(
    all_labels,
    all_predictions
)


print("\n================================")
print("FINAL EVALUATION")
print("================================")

print(f"Test Accuracy: {accuracy * 100:.2f}%")


# ============================================================
# 13. CLASSIFICATION REPORT
# ============================================================

classes = (
    train_dataset.dataset.classes
    if isinstance(train_dataset, torch.utils.data.Subset)
    else train_dataset.classes
)

print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=classes
    )
)


# ============================================================
# 14. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

plt.figure(figsize=(7, 6))

plt.imshow(cm)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.colorbar()

plt.xticks(
    range(len(classes)),
    classes,
    rotation=45
)

plt.yticks(
    range(len(classes)),
    classes
)

for i in range(len(classes)):

    for j in range(len(classes)):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()

plt.show()


# ============================================================
# 15. TRAINING LOSS VS VALIDATION LOSS
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    train_losses,
    marker="o",
    label="Training Loss"
)

plt.plot(
    val_losses,
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training and Validation Loss")

plt.legend()

plt.grid()

plt.show()


# ============================================================
# 16. TRAINING ACCURACY VS VALIDATION ACCURACY
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    train_accuracies,
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    val_accuracies,
    marker="o",
    label="Validation Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy (%)")

plt.title("Training and Validation Accuracy")

plt.legend()

plt.grid()

plt.show()


# ============================================================
# 17. DISPLAY AUGMENTED IMAGES
# ============================================================

images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 6))

for i in range(10):

    plt.subplot(2, 5, i + 1)

    image = images[i].permute(1, 2, 0).numpy()

    image = image * np.array(
        [0.229, 0.224, 0.225]
    )

    image = image + np.array(
        [0.485, 0.456, 0.406]
    )

    image = np.clip(image, 0, 1)

    plt.imshow(image)

    plt.title(classes[labels[i]])

    plt.axis("off")

plt.suptitle("Augmented Training Images")

plt.tight_layout()

plt.show()


# ============================================================
# 18. DISPLAY PREDICTIONS
# ============================================================

images, labels = next(iter(test_loader))

images_gpu = images.to(device)

with torch.no_grad():

    outputs = model(images_gpu)

    predictions = torch.argmax(
        outputs,
        dim=1
    )


plt.figure(figsize=(12, 6))

for i in range(10):

    plt.subplot(2, 5, i + 1)

    image = images[i].permute(1, 2, 0).numpy()

    image = image * np.array(
        [0.229, 0.224, 0.225]
    )

    image = image + np.array(
        [0.485, 0.456, 0.406]
    )

    image = np.clip(image, 0, 1)

    plt.imshow(image)

    plt.title(
        f"Actual: {classes[labels[i]]}\n"
        f"Pred: {classes[predictions[i].cpu()]}"
    )

    plt.axis("off")

plt.tight_layout()

plt.show()


# ============================================================
# 19. FINAL OBSERVATION
# ============================================================

print("\n================================")
print("OBSERVATION")
print("================================")

print(
    "Data augmentation was applied using random horizontal "
    "and vertical flipping, rotation and color variation."
)

print(
    "These transformations create varied training examples "
    "and help the model generalize better to unseen images."
)

print(
    f"Final Test Accuracy: {accuracy * 100:.2f}%"
)

print(
    "The loss and accuracy graphs show the training progress "
    "and the confusion matrix shows classification performance "
    "for the three bean classes."
)


Device: cpu
